# Day 8 — Git Properly: Commits, Branches, Pull Requests & Rebase (1 Nov)

*Rebuilt from the master Google Doc session (lesson text and every code example). Code cells are not executed: several need Postgres, a running server or files from the original starter zip.*

## Today at a glance

- Goal: use Git the way a team does: small commits with clear messages, a .gitignore that keeps secrets and junk out, a branch for every change, pull requests for review, and rebase to keep a branch current. You practise all of it on your own code by bringing Days 5 and 6 into main.

- Deliverable: two merged pull requests on GitHub (PR 1: day05/, PR 2: day06/), an improved .gitignore, and day08/repo_check.py plus day08/dsa.py from today's starters.

- Where this sits: Week 2, Phase 1 (Foundation), day 1 of 7. From today every change in the repo goes through a branch and a pull request, the same workflow you will meet in every customer repo.

Time split (5h):

| Block | Time | What you do |
|---|---|---|
| Theory | 1.5h | Concepts 1 to 6, typing every command in a throwaway demo repo |
| Hands-on | 2.5h | PR 1 (Day 5), PR 2 (Day 6) with a rebase and a conflict, .gitignore, repo_check.py |
| DSA | 0.5h | Merge two sorted lists, find the first bad commit |
| Revision | 0.5h | Quiz, Remember cells in the notebook, open items |

Open items first: the repo has Days 1 to 3 finished. day04/ still holds the starter TODOs, and there is no day05/, day06/ or day07/ folder yet. Your commit on main is named "Day 4" and its author email is rohitkini@Rohits-MacBook-Air.local, a laptop address GitHub cannot link to your profile, so the commit shows no avatar and does not count on your contribution graph. Today fixes both problems: the email in Step 0 and the missing days in Steps 1 to 3.

## Recap (2 minutes, from memory)

- What do the two print lines below show? Where should the real key live so that it can never reach GitHub?

- Day 7 named the five steps every integration goes through. Name them, and the Week 1 day that taught each one.

- Day 7 set a rule for pushing. What is it, and why does unpushed work count as not done?

Predict first, then run it:

In [ ]:
# Recap: predict both lines, then run. Day 6's secrets meet today's Git.
import os

os.environ.pop("CMS_API_KEY", None)  # make sure the key is not set
print(os.environ.get("CMS_API_KEY", "missing"))
# set the key for this Python process only (like process.env in Node)
os.environ["CMS_API_KEY"] = "demo-key-123"
print(os.environ.get("CMS_API_KEY", "missing")[:4] + "...")

Answers are on the last page of today's session.

## Concept 1: How Git thinks, snapshots and three areas

### Why it matters for an FDE

In a customer engagement you work inside their repository, next to their engineers, often with strict review rules. You must be able to say exactly what changed, undo a change without breaking anything, and hand over a history that someone else can follow after you leave. Your public repo is also read by hiring managers: a clean history is evidence you have worked on a team.

### How it works

A commit is a saved snapshot of every tracked file in the project, plus a message, an author, a time and a pointer to the commit before it (its parent). Git names each commit by a hash of all that, such as f8b34fd. Because the hash depends on the content and the parent, a commit can never change; "changing" one always makes a new commit with a new hash.

Your work moves through three areas. The working tree is the files you edit in VS Code. The staging area (also called the index) is the next commit you are building: git add copies a file's current content there. The repository is the .git folder holding every commit: git commit turns the staging area into a new snapshot. In React terms, the staging area is the new state object you build up before calling setState, and a commit is a state value you never mutate. In Contentstack terms, every commit is an entry version you can compare and roll back to.

```bash
# Concept 1: watch one file move through the three areas
mkdir git-demo && cd git-demo  # a throwaway folder, outside fde-journey
git init -b main  # new repository; -b main names the first branch
echo 'print("v1")' > ticket.py  # create a file in the working tree
git status --short
# -> ?? ticket.py      (?? = untracked: Git does not know this file yet)
git add ticket.py  # copy the file's current content into staging
git status --short
# -> A  ticket.py      (A in the left column = new file, staged)
git commit -m "Add ticket script"  # staging becomes a snapshot
# -> [main (root-commit) f8b34fd] Add ticket script
echo 'print("v2")' > ticket.py  # edit the file again
git status --short
# -> " M ticket.py"    (M in the RIGHT column = changed, not staged)
```

Read git status --short as two columns: the left one is the staging area, the right one is the working tree. M  (left) means "changed and staged",  M (right) means "changed but not staged yet". Now compare what each diff shows:

```bash
# Concept 1: diff, diff --staged, log and restore
git diff  # working tree vs staging: what you have NOT added yet
# -> -print("v1")
# -> +print("v2")
git add ticket.py
git diff  # prints nothing: working tree and staging now match
git diff --staged  # staging vs last commit: what WILL be committed
# -> -print("v1")
# -> +print("v2")
git commit -m "Print v2 from ticket script"
git log --oneline  # newest first, one line per commit
# -> 0013dac Print v2 from ticket script
# -> f8b34fd Add ticket script
echo 'oops' >> ticket.py  # a change you regret
git restore ticket.py  # put the file back to its staged/committed state
cat ticket.py
# -> print("v2")
# GOTCHA: restore throws away uncommitted edits for good. No undo.
# REMEMBER: add = put it in the next snapshot; commit = save the snapshot
```

Your hashes will be different from the ones shown, because the hash includes your name, email and the exact time.

### Real-world FDE example

A customer says "the nightly export broke after Tuesday's change". You run git log --oneline --since="last monday", find the three commits from that week, and git show <hash> prints exactly which lines each one changed. Ten minutes later you can tell them which change caused it and revert only that one. Without history, the same question takes a day of guessing.

### Common mistakes

- Thinking a commit sends your work to GitHub. It does not: commits stay on your laptop until you git push.

- Running git add . without reading git status first, which stages everything, including .env and output files.

- Using git restore on a file with work you wanted to keep. Commit or copy it first.

- Editing a copy of the project in Downloads instead of the cloned repo, then wondering why git status shows nothing.

## Concept 2: Commits that tell a story

### Why it matters for an FDE

Reviewers, the customer's engineers and you in three months will read your history to understand why the code looks the way it does. Small commits with clear messages make a pull request easy to review and make it safe to undo one change without losing the others. A history of "Day 4", "fix" and "changes" tells the reader nothing.

### How it works

Make each commit one logical change: the dataclass, then its validation, then the loader, not "everything I did today". A good test: can you describe the commit in one short sentence without the word "and"?

The message has a subject line of at most 50 characters, in the imperative mood ("Add retry", not "Added retry"), starting with a capital letter and with no full stop. It should complete the sentence "If applied, this commit will ...". When the reason is not obvious, add a blank line and a short body that explains why.

| Weak message | Better message | Why it is better |
|---|---|---|
| Day 4 | Add JSON contact store with atomic save | Says what changed, not when |
| fixed stuff | Fix KeyError when a contact has no email | Names the bug that is gone |
| Updated files | Ignore .env and generated output folders | Imperative and specific |
| WIP | WIP: parse log lines, summary still pending | WIP is fine when it says what is left |

To commit only part of a file, use git add -p: Git shows each changed block (a hunk) and asks whether to stage it. To fix the last commit before you push it, use git commit --amend.

```bash
# Concept 2: stage part of a file, write a message with a body, amend
git add -p support_ticket.py  # y = stage this hunk, n = skip, q = quit
git diff --staged --stat  # check what the commit will contain
git commit  # no -m: opens your editor for a subject AND a body
# Fixed a typo in the subject? Rewrite the last commit's message:
git commit --amend -m "Validate ticket email in __post_init__"
# GOTCHA: --amend makes a NEW commit (new hash). Only amend commits you
# have not pushed yet, or the next push is rejected (see Concept 6).
# FDE TIP: run "git config --global core.editor 'code --wait'" once, so
# "git commit" opens VS Code instead of vim.
```

This is what a commit with a body looks like in the editor. The first line is the subject, then one blank line, then the body:

```bash
Add retry with backoff to fetch_page

The customer's CMS returns 503 under load. Retry 429 and 5xx up to
3 times (0.5s, 1s, 2s) and honour Retry-After when it is sent.
401 and 422 still fail at once, because repeating them cannot work.
```

### Real-world FDE example

During a security review, a customer asks when PII masking was added to the export and why. git log --grep="mask" --oneline finds the commit in seconds, and its body explains that the customer's DPO asked for email addresses to be masked in logs. Many customer repos also require a message convention such as Conventional Commits (feat: ..., fix: ...). Read the repo's last 20 commits before your first one and match what you see.

### Common mistakes

- One giant commit per day. Nobody can review it, and nobody can revert half of it.

- Messages that say how ("changed if to elif") instead of what and why.

- Committing debug print() lines and commented-out code. Read git diff --staged before every commit.

- Amending a commit that is already pushed and shared.

## Concept 3: .gitignore and secrets

### Why it matters for an FDE

Pushing a customer's API key to a public repo is one of the worst mistakes an FDE can make: bots scan GitHub for keys within minutes, and the customer's security team will ask how it happened. A .gitignore also keeps out files that are rebuilt on every machine (.venv/, __pycache__/) and output your code produces, so the repo holds only what a person wrote.

### How it works

.gitignore is a text file at the repo root with one pattern per line. Git hides matching untracked files from git status and git add .. You already know the idea: every React project ignores node_modules/ and .env.local.

- .env matches a file or folder with that name in any folder.

- output/ (with a slash) matches folders only, in any folder.

- *.pyc uses a wildcard: any name ending in .pyc.

- /build (leading slash) matches only at the repo root.

- !sample.csv is an exception: track this file even though an earlier pattern ignores it.

- Lines starting with # are comments.

Here is a full .gitignore for this repo, rule by rule:

```bash
# Concept 3: a .gitignore for fde-journey (save it at the repo root)
# Python byproducts, rebuilt automatically on every machine
__pycache__/
*.pyc
.ipynb_checkpoints/
# Virtual environments, like node_modules: rebuilt from requirements.txt
.venv/
# Secrets: never commit them. The real key lives only on your machine.
.env
# Output your scripts generate (Day 4 store, Day 6 CSVs)
output/
day04/data/
# Tool caches you will meet on Days 10 and 12
.pytest_cache/
.mypy_cache/
.ruff_cache/
# macOS folder metadata
.DS_Store
```

Two commands help you check the rules. git check-ignore -v tells you which line ignores a file. git rm --cached stops tracking a file that was committed before you ignored it, without deleting it from your disk, because .gitignore only affects files Git is not tracking yet.

```bash
# Concept 3: check the rules, and untrack a file committed by mistake
git status --short  # before the .gitignore exists
# -> ?? .env
# -> ?? __pycache__/
# -> ?? day06/
git status --short  # after saving a .gitignore with the rules above
# -> ?? .gitignore     (.env, __pycache__ and output are now hidden)
git check-ignore -v .env day06/output/posts.csv
# -> .gitignore:5:.env      .env
# -> .gitignore:7:output/   day06/output/posts.csv
# (file:line:pattern, then the path; line numbers are from the demo file)
git check-ignore -v README.md  # prints nothing, exit code 1: not ignored
# config.local was committed BEFORE it was added to .gitignore:
git status --short
# -> " M config.local"   (still tracked: ignoring came too late)
git rm --cached config.local  # untrack it; the file stays on disk
git status --short
# -> D  config.local    (the next commit removes it from the repo)
```

GOTCHA: deleting a committed secret in a new commit does not remove it. The old commit still holds it, and anyone can read the history. Treat the key as leaked: revoke it and create a new one first, then clean the history (with git filter-repo) only if the customer asks. GitHub's push protection blocks some known key formats, but never rely on it.

### Real-world FDE example

A migration script for a Contentstack stack needs a management token, which can create and delete content. The token goes in .env (ignored), the script reads it with os.environ["CS_MANAGEMENT_TOKEN"], and the repo gets a committed .env.example with the variable names and fake values, so the next engineer knows what to set. If the token ever reaches a commit, the first action is revoking it in the stack settings, before anything else.

### Common mistakes

- Adding .env to .gitignore after it was already committed, and assuming that removed it.

- Ignoring files that should be committed: requirements.txt, .env.example, or test data such as tickets_export.json.

- Broad patterns like *.json that silently hide data files the code needs. Prefer folder rules such as output/.

- Naming the file gitignore or .gitignore.txt. It must be exactly .gitignore.

## Concept 4: Branches and merging

### Why it matters for an FDE

At a customer you never commit straight to main: main is what gets deployed. A branch lets you build a change in isolation, keep main working, and switch to an urgent hotfix while a feature is half done. Every pull request starts with a branch.

### How it works

A branch is only a label that points at a commit, the same idea as Day 1's "variables are labels, not boxes". HEAD is a second label saying which branch you are on. When you commit, the branch you are on moves forward to the new commit; the others stay where they were. Creating a branch costs nothing: Git writes one small file holding a hash. Contentstack's own Branches feature works the same way for content models: you change a copy in isolation and merge it back.

Merging brings a branch's commits into another branch. If main has not moved since you branched, Git does a fast-forward: it slides the main label up to your last commit, and no new commit is made. If both branches have new commits, Git makes a merge commit with two parents. If both changed the same or neighbouring lines, Git cannot choose and reports a conflict for you to resolve.

```bash
# Concept 4: a branch is a label; a fast-forward merge just moves it
git branch  # list branches; * marks the one you are on
# -> * main
git switch -c day05-ticket  # create a branch AND switch to it
# -> Switched to a new branch 'day05-ticket'
echo 'class Ticket: ...' > ticket.py
git add ticket.py && git commit -m "Add Ticket class"
git log --oneline --graph --all --decorate  # labels in brackets
# -> * 265bc0f (HEAD -> day05-ticket) Add Ticket class
# -> * c26c6e1 (main) Add README
git switch main  # ticket.py disappears from the folder: main lacks it
git merge day05-ticket  # main has not moved, so this is a fast-forward
# -> Fast-forward
git log --oneline --decorate
# -> 265bc0f (HEAD -> main, day05-ticket) Add Ticket class
# -> c26c6e1 Add README
git branch -d day05-ticket  # delete the label; the commits stay
```

Now a conflict. Two branches both change the Day 6 row of the README, which is exactly what will happen in today's task:

```bash
# Concept 4: both branches edit the same line, so Git asks you to choose
git merge day06-export
# -> CONFLICT (content): Merge conflict in README.md
git status --short
# -> UU README.md      (UU = unmerged: both sides changed it)
cat README.md
# -> | Day | Files |
# -> | 5 | - |
# -> <<<<<<< HEAD
# -> | 6 | export.py, dsa.py |
# -> =======
# -> | 6 | cms_export.py |
# -> >>>>>>> day06-export
# Between <<<<<<< and ======= is your current branch (HEAD); between
# ======= and >>>>>>> is the branch coming in. Edit the file so it holds
# the final text you want, delete all three marker lines, then:
git add README.md  # tells Git "this conflict is resolved"
git commit --no-edit  # finishes the merge with the default message
git log --oneline --graph --decorate
# -> *   b4f2d67 (HEAD -> main) Merge branch 'day06-export'
# -> |\
# -> | * 0075758 (day06-export) Fill Day 6 row in README
# -> * | e988eb2 List Day 6 files
# -> |/
# -> * 265bc0f Add Ticket class
# REMEMBER: a branch is a movable label; merging moves or joins labels
```

VS Code shows conflicts with buttons above each block (Accept Current, Accept Incoming, Accept Both). They are handy, but always read the result: "Accept Both" often leaves two versions of the same row.

### Real-world FDE example

You are two days into a feature branch for a customer's search connector when their content team reports that the export drops entries with emoji in the title. You commit your work in progress, git switch main, create fix-emoji-export, fix and ship it through its own pull request, then switch back. The feature work never got mixed into the hotfix.

### Common mistakes

- Committing on main by accident. Check the first line of git status ("On branch ...") before you start work.

- Keeping a branch open for weeks: the further main moves, the bigger the conflicts.

- Resolving a conflict by deleting the markers but leaving both versions of the line.

- Switching branches with uncommitted changes and carrying them along by surprise. Commit (or git stash) first.

## Concept 5: Remotes and pull requests

### Why it matters for an FDE

A pull request (PR) is how code gets reviewed and accepted at almost every company, and at a customer it is often the only way your code reaches main. The PR description is also customer-facing writing: the customer's engineer decides from it whether to trust your change. Clear, small PRs get merged in hours; big unclear ones wait for weeks.

### How it works

A remote is another copy of the repository, usually on GitHub, named origin by default. origin/main is your laptop's last known view of GitHub's main; it only updates when you run git fetch (or git pull, which is fetch plus merge). git push -u origin <branch> uploads a branch and remembers the link, so later a plain git push works.

A pull request asks to merge one branch into another on GitHub. It shows the diff, runs any automated checks, and holds the review discussion. When it is approved, GitHub merges it in one of three ways:

| Merge button | What main gets | Use it when |
|---|---|---|
| Create a merge commit | All your commits plus one merge commit | You want the branch history kept as it was |
| Squash and merge | One commit holding the whole PR | The branch has messy WIP commits |
| Rebase and merge | Your commits replayed in a straight line | Your commits are clean and you want linear history |

Here is the push step against a local stand-in for GitHub, so you can see real output. The gh commands after it are GitHub's own command-line tool; their output depends on your account, so it is not shown.

```bash
# Concept 5: a remote, and pushing a branch to it
git remote -v  # where "origin" points (yours: github.com/rohitkini34-byte)
# -> origin  .../origin.git (fetch)
# -> origin  .../origin.git (push)
git switch -c day06-export
git push -u origin day06-export  # -u links local and remote branch
# -> * [new branch]      day06-export -> day06-export
# -> branch 'day06-export' set up to track 'origin/day06-export'.
```

```bash
# Concept 5: pull requests from the terminal with GitHub's gh tool
brew install gh  # once (Windows: winget install GitHub.cli)
gh auth login  # once: choose GitHub.com, HTTPS, log in with a browser
gh pr create --draft --title "Day 6: CMS export CLI" --body-file pr.txt
# --draft = "not ready yet, but look at the direction"
gh pr ready  # mark the draft as ready for review
gh pr view --web  # open the PR page in your browser
gh pr merge --merge --delete-branch  # merge commit + delete the branch
# FDE TIP: the same steps work in the GitHub web page. Use whichever you
# like, but always read the "Files changed" tab before you merge.
```

A PR description needs four short parts. Keep this as pr.txt and fill it in for every PR:

```bash
What: one or two sentences on what this PR changes.
Why: the problem it solves or the task it completes.
How to test: the exact commands a reviewer runs, and what they should see.
Open items: anything not finished yet, or "none".
```

### Real-world FDE example

A customer's main is protected: every PR needs one approval from their team and a green test run. On day one you open a draft PR with only the folder layout and a description of the plan. Their engineer comments that the CRM IDs must stay strings, not integers. You learn it before writing 300 lines on the wrong assumption.

### Common mistakes

- Huge PRs. Over about 400 changed lines, reviewers skim instead of reading. Split the work.

- Branching from an old main. Run git switch main && git pull before creating a branch.

- An empty description, or "see code". The reviewer should not have to guess how to test it.

- Merging your own PR without reading its diff in "Files changed".

## Concept 6: Rebase, and how to recover from mistakes

### Why it matters for an FDE

While your PR waits for review, main keeps moving. Reviewers and many customer repos ask you to "rebase on main" so your branch includes the latest code and the history stays a straight line. Knowing how to recover from a bad rebase or reset is what lets you use Git without fear.

### How it works

git rebase origin/main takes the commits that exist only on your branch and replays them, one by one, on top of the latest origin/main. The result looks as if you had started your branch today. Because each replayed commit gets a new parent, each one gets a new hash: these are new commits with the same changes.

That is why a rebased branch that was already pushed needs git push --force-with-lease. A plain push is rejected because the branch on GitHub has commits your branch no longer has. --force-with-lease overwrites it only if nobody else pushed there since your last fetch; plain --force overwrites whatever is there.

```bash
# Concept 6: main moved while your branch waited; rebase onto it
git fetch origin  # download new commits; your files do not change
# ->    576becd..4aed7ba  main       -> origin/main
git log --oneline --graph --all --decorate
# -> * a8e048e (HEAD -> day06-export, origin/day06-export) Add cms_ex...
# -> | * 4aed7ba (origin/main) Add Ticket class (#1)
# -> |/
# -> * 576becd (main) Add README
git rebase origin/main  # replay my commit on top of the new main
# -> Successfully rebased and updated refs/heads/day06-export.
git log --oneline --graph --all --decorate
# -> * 0cfe9ef (HEAD -> day06-export) Add cms_export CLI skeleton
# -> * 4aed7ba (origin/main) Add Ticket class (#1)
# -> | * a8e048e (origin/day06-export) Add cms_export CLI skeleton
# -> |/
# -> * 576becd (main) Add README
# Same change, new hash: a8e048e became 0cfe9ef.
git push
# -> ! [rejected]        day06-export -> day06-export (non-fast-forward)
git push --force-with-lease  # safe overwrite of MY branch on GitHub
# -> + a8e048e...0cfe9ef day06-export -> day06-export (forced update)
```

When a replayed commit conflicts, the rebase pauses on that commit. You fix the file exactly as in Concept 4, then git add it and run git rebase --continue (not git commit). If it goes wrong, git rebase --abort puts everything back as it was before you started.

The golden rule: only rebase commits that are yours alone. Never rebase main or a branch other people have pulled, because their copies would still hold the old commits.

|  | git merge | git rebase |
|---|---|---|
| History | Keeps every branch and adds a merge commit | One straight line, no merge commit |
| Hashes | Existing commits keep their hashes | Replayed commits get new hashes |
| Safe on shared branches | Yes | No: only on your own branch |
| Typical use | Bringing a finished branch into main | Updating your branch with the latest main |

Your safety net is the reflog, a private list of every place HEAD has pointed recently, kept for about 90 days. Even commits that seem lost after a rebase or reset are still there:

```bash
# Concept 6: the reflog remembers where HEAD has been
git reflog -n 4  # newest first
# -> 0cfe9ef HEAD@{0}: rebase (finish): returning to refs/heads/day06-export
# -> 0cfe9ef HEAD@{1}: rebase (pick): Add cms_export CLI skeleton
# -> 4aed7ba HEAD@{2}: rebase (start): checkout origin/main
# -> a8e048e HEAD@{3}: commit: Add cms_export CLI skeleton
git branch rescue a8e048e  # put a label on the old commit to keep it
# REMEMBER: rebase = same changes, new commits; never rebase shared work
```

git rebase -i origin/main (interactive) opens a list of your commits in the editor, where you can squash small fix-up commits into the one they fix before review. Use it on your own branch only, and you will use it properly from Day 12 onward.

### Real-world FDE example

Your PR into a customer's repo has waited two days, and meanwhile their team merged a security fix that changes the same config file. The reviewer writes "please rebase on main". You fetch, rebase, resolve one conflict in the config, run the tests again, and force-push with lease. The PR now shows only your changes, on top of their fix.

### Common mistakes

- Using git push --force instead of --force-with-lease, and wiping out a teammate's commit.

- Rebasing main itself, or a branch someone else works on.

- Running git commit in the middle of a rebase instead of git rebase --continue.

- Deleting the folder and cloning again after a mistake. That loses local work; git rebase --abort or the reflog fixes it.

## Check your understanding (predict first, then run)

Do these in your throwaway git-demo folder, not in fde-journey. Write each prediction down before you press Enter.

```bash
# Check 1: what does the last line print?
echo x > notes.txt
git add notes.txt
echo y >> notes.txt  # change the file AFTER staging it
git status --short

# Check 2: which of the three paths are ignored?
printf '*.csv\n!sample.csv\n' > .gitignore  # two rules, one exception
mkdir -p out && touch data.csv sample.csv out/report.csv
git check-ignore data.csv sample.csv out/report.csv

# Check 3: a branch with two commits, then main gets one more commit
git switch -c feature
echo 1 > f1 && git add f1 && git commit -m "Add f1"
echo 2 > f2 && git add f2 && git commit -m "Add f2"
git log --oneline main..HEAD  # commits on HEAD that main does not have
git switch main
echo m > m && git add m && git commit -m "Add m"
git switch feature
git rebase main
git log --oneline main..HEAD  # how many lines now? same hashes?
```

- Check 1: what are the two letters in front of notes.txt, and what does each column mean here?

- Check 2: which paths does check-ignore print, and why is one missing?

- Check 3: how many commits does the second main..HEAD log show, and are their hashes the same as in the first one?

- Without running it: in Check 3, what would git push say if feature had been pushed before the rebase, and what is the safe fix?

## Hands-on task (about 2.5 hours): Days 5 and 6 into main through two pull requests

Treat fde-journey like a customer repo: nobody commits to main directly; every change arrives through a branch and a reviewed PR. You are the author today, and also the reviewer.

Before you start: the Day 5 and Day 6 files come from those days' sessions: the starter files that came with them (support_ticket.py, tickets_export.json, mock_cms.py, cms_export.py and both dsa.py files) plus whatever you have written. If you no longer have the starters, say so in the project chat before you begin. Unfinished code is fine: commit what you have and list what is missing under Open items. Finishing the code is not today's goal; getting it reviewed and merged properly is.

Step 0, setup (15 min). Fix your identity so new commits link to your GitHub profile, install gh, and update main (it now holds today's day08/ starters).

```bash
# Step 0: run once on your Mac
git config --global user.name "Rohit Kini"
# Use the email on your GitHub account, or the private noreply address
# from GitHub > Settings > Emails (it looks like 12345+name@users...)
git config --global user.email "you@example.com"
git config --global core.editor "code --wait"  # commit messages in VS Code
brew install gh && gh auth login
cd ~/path/to/fde-journey  # your clone
git switch main && git pull  # get today's day08/ folder
git log -1 --format="%an <%ae>"  # the old commit keeps the old email
```

Step 1, PR 1: Day 5 (45 min).

- From an up-to-date main, create day05-support-ticket.

- Add day05/ in at least three commits, one logical piece each (for example: the starter files and data; your dataclass and validation; the queue, loader and dsa.py). Fill row 5 of the README's Files column in a commit of its own.

- Push with -u and open the PR with a filled-in pr.txt.

- Do not merge it yet.

Step 2, PR 2: Day 6 (45 min).

- Switch back to main (PR 1 is not merged, so main has no day05/) and create day06-cms-export from it.

- First commit: extend the repo's .gitignore with the missing rules from Concept 3 (.env, output/, day04/data/ and the caches). Check it with git check-ignore -v.

- Add day06/ in at least three commits, with requirements.txt but never .venv/, .env or output/. Fill README row 6 in its own commit.

- Push and open PR 2. If cms_export.py is not finished, open it as a draft.

Step 3, merge, rebase, conflict (45 min).

- On GitHub, read PR 1's "Files changed" as a reviewer, leave yourself at least one comment, then merge it with "Create a merge commit" and delete the branch.

- Locally: git switch main && git pull, then switch to day06-cms-export and git rebase origin/main. The README rows 5 and 6 sit next to each other, so expect a conflict. Resolve it so both rows are filled, git add README.md, git rebase --continue.

- Push with --force-with-lease. Mark PR 2 ready (if it was a draft and the code works), then merge it with "Rebase and merge" and delete the branch.

- git switch main && git pull, then git branch -d both local branches.

Step 4, repo_check.py (30 min). On a branch day08-repo-check, complete day08/repo_check.py and day08/dsa.py from the starters, run python day08/repo_check.py from the repo root, and bring them in through a third, small PR.

Acceptance criteria:

- GitHub shows PR 1 (Day 5) and PR 2 (Day 6) as merged, each with a description that fills What, Why, How to test and Open items.

- Each of the two PRs has at least three commits, and every new subject passes repo_check.py's rules: 50 characters or fewer, capital first letter, imperative, no full stop, at least three words.

- git log --oneline --graph on main shows PR 1's merge commit, with PR 2's commits after it in a straight line.

- git ls-files | grep -E "(^|/)(\.env|\.venv|__pycache__|output)(/|$)" prints nothing, and git check-ignore -v .env prints the rule that ignores it.

- README rows 5 and 6 list their files, and git grep -n "<<<<<<<" prints nothing (no conflict markers left anywhere).

- Your new commits show your GitHub avatar, because the author email matches your account.

- python day08/repo_check.py prints the report and flags your old "Day 4" subject as too short, and nothing you committed today.

- git status on main says "Your branch is up to date with 'origin/main'", and the merged branches are deleted locally and on GitHub.

Hints (no solutions, try first):

- Before every commit, run git status and git diff --staged. If you see .venv, .env or output in the list, stop and fix .gitignore first.

- To make three commits from files that already exist, stage them in groups: git add day05/tickets_export.json day05/dsa.py, commit, then the next group. Use git add -p when one file holds two logical pieces.

- If the rebase conflict confuses you, run git status: it names the file and tells you the next command. git rebase --abort is always a safe way back.

- If a push is rejected after the rebase, that is expected; read Concept 6 again before you reach for a force push.

- repo_check.py uses subprocess.run, which is new; the starter's hints show the exact call. It runs git the same way you type it in the terminal and hands back the printed text.

- If you commit to main by accident before pushing: git branch my-work keeps the commit on a new label, then git reset --hard origin/main moves main back. Check git log --oneline --all before and after.

## DSA practice (30 min)

Both problems are in day08/dsa.py, with signatures, examples and hints.

- Merge two sorted lists. merge_sorted(a, b) returns one sorted list without calling sorted(). merge_sorted([1, 4, 9], [2, 3, 10, 12]) gives [1, 2, 3, 4, 9, 10, 12]. Hint: keep one index into each list, always take the smaller front item, and think about what is left over when one list runs out. This is the core step of merge sort, and the same "walk two sorted streams" idea Git uses to compare histories.

- First bad commit. first_bad(versions, is_bad) returns the first version for which is_bad(version) is True, or None, calling is_bad as few times as possible. It is what git bisect does to find the commit that broke a build. Hint: once a version is bad, every later one is bad too, so check the middle and throw away half the list each time (binary search). Eight versions need at most four checks.

## Quiz (3 quick questions)

- A teammate pushed .env with a customer's CMS key to a public repo, then removed it in the next commit. Is the key safe now? What do you do, in what order?

- When would you merge main into your branch, and when would you rebase your branch onto main? State the golden rule.

- Explain to a customer's non-technical project manager, in two sentences, what a pull request is and why you open one before changing their system.

## What you achieved today

You learned how Git stores work (snapshots, three areas, branches as labels), wrote commits other people can read, kept secrets and generated files out with .gitignore, and ran the full team workflow: branch, push, pull request, review, merge, rebase and conflict resolution. You also brought two missing days into the repo, so the next weekly score reads more of your real work.

Tomorrow (Day 9): the terminal and shell for engineers (paths, pipes, grep, environment variables, exit codes, processes and ports), building log_report.py, a CLI that reads a server log from stdin and prints an error summary.

## When you're done

Paste into the Forward Deployment Engineer project chat as "Day 8 review": the links to your two PRs, the output of git log --oneline --graph -n 15 on main, your final .gitignore, your recap and check-your-understanding predictions next to the real outputs, repo_check.py, dsa.py and your quiz answers. Say which step was hardest; that is where tomorrow's recap will start.

## Recap answers

*Your answers:*
